In [23]:
import pandas as pd

# 파일 경로 설정
base_path = "../data/"  # 필요시 경로 수정
orders = pd.read_csv(base_path + "olist_orders.csv")
order_items = pd.read_csv(base_path + "olist_order_items.csv")
customers = pd.read_csv(base_path + "olist_customers.csv")

# 1. 고객 ID 기준으로 주문 데이터와 결합
merged_orders = pd.merge(orders, customers, on='customer_id', how='left')

# 2. 진짜 고객 기준 (customer_unique_id)으로 주문 수 계산
customer_order_counts = merged_orders.groupby('customer_unique_id').agg(
    order_count=('order_id', 'nunique'),
    customer_state=('customer_state', 'first')
).reset_index()

# 3. 재구매 여부 추가 (주문 2개 이상이면 재구매)
customer_order_counts['is_repeat'] = customer_order_counts['order_count'].apply(lambda x: 1 if x >= 2 else 0)

# 4. 지역별 재구매율 요약 (Chart 1용)
state_repeat_summary = customer_order_counts.groupby('customer_state').agg(
    total_customers=('customer_unique_id', 'count'),
    repeat_customers=('is_repeat', 'sum')
).reset_index()
state_repeat_summary['repeat_rate'] = (
    state_repeat_summary['repeat_customers'] / state_repeat_summary['total_customers']
).round(4)

# 5. 주문별 총 지출 계산
order_items['total_price'] = order_items['price'] + order_items['freight_value']
order_total = order_items.groupby('order_id')['total_price'].sum().reset_index()

# 6. 주문에 총 지출 조인
orders_with_total = pd.merge(orders, order_total, on='order_id', how='left')

# 7. 고객 ID 붙이기
orders_with_customer = pd.merge(
    orders_with_total,
    customers[['customer_id', 'customer_state', 'customer_unique_id']],
    on='customer_id',
    how='left'
)

# 8. 고객별 평균 주문 수 & 지출 계산 (Chart 2용)
customer_summary = orders_with_customer.groupby('customer_unique_id').agg(
    avg_order_count=('order_id', 'nunique'),
    total_spent=('total_price', 'sum'),
    customer_state=('customer_state', 'first')
).reset_index()
customer_summary['avg_total_spent'] = (
    customer_summary['total_spent'] / customer_summary['avg_order_count']
).round(2)

# 9. 파일 저장
state_repeat_summary.to_csv("state_repeat_summary.csv", index=False)
customer_summary.to_csv("customer_summary.csv", index=False)

# ✅ Chart 2를 위한 state 단위 요약
state_customer_behavior = customer_summary.groupby('customer_state').agg(
    avg_order_count=('avg_order_count', 'mean'),
    avg_total_spent=('avg_total_spent', 'mean')
).reset_index()

# 파일 저장
state_customer_behavior.to_csv("state_customer_behavior.csv", index=False)



In [ ]:
# import pandas as pd

# # 파일 경로 설정
# base_path = "../data/"  # 필요시 경로 수정
# orders = pd.read_csv(base_path + "olist_orders.csv")
# order_items = pd.read_csv(base_path + "olist_order_items.csv")
# customers = pd.read_csv(base_path + "olist_customers.csv")

# # 1. 고객 ID 기준으로 주문 데이터와 결합
# merged_orders = pd.merge(orders, customers, on='customer_id', how='left')

# # 2. 진짜 고객 기준 (customer_unique_id)으로 주문 수 계산
# customer_order_counts = merged_orders.groupby('customer_unique_id').agg(
#     order_count=('order_id', 'nunique'),
#     customer_state=('customer_state', 'first')
# ).reset_index()

# # 3. 재구매 여부 추가 (주문 2개 이상이면 재구매)
# customer_order_counts['is_repeat'] = customer_order_counts['order_count'].apply(lambda x: 1 if x >= 2 else 0)

# # 4. 지역별 재구매율 요약 (Chart 1용)
# state_repeat_summary = customer_order_counts.groupby('customer_state').agg(
#     total_customers=('customer_unique_id', 'count'),
#     repeat_customers=('is_repeat', 'sum')
# ).reset_index()
# state_repeat_summary['repeat_rate'] = (
#     state_repeat_summary['repeat_customers'] / state_repeat_summary['total_customers']
# ).round(4)

# # 5. 주문별 총 지출 계산
# order_items['total_price'] = order_items['price'] * order_items['freight_value']
# order_total = order_items.groupby('order_id')['total_price'].sum().reset_index()

# # 6. 주문에 총 지출 조인
# orders_with_total = pd.merge(orders, order_total, on='order_id', how='left')

# # 7. 고객 ID 붙이기
# orders_with_customer = pd.merge(
#     orders_with_total,
#     customers[['customer_id', 'customer_state', 'customer_unique_id']],
#     on='customer_id',
#     how='left'
# )

# # 8. 고객별 평균 주문 수 & 지출 계산 (Chart 2용)
# customer_summary = orders_with_customer.groupby('customer_unique_id').agg(
#     avg_order_count=('order_id', 'nunique'),
#     total_spent=('total_price', 'sum'),
#     customer_state=('customer_state', 'first')
# ).reset_index()
# customer_summary['avg_total_spent'] = (
#     customer_summary['total_spent'] / customer_summary['avg_order_count']
# ).round(2)

# # 9. 파일 저장
# state_repeat_summary.to_csv("state_repeat_summary.csv", index=False)
# customer_summary.to_csv("customer_summary.csv", index=False)
